# HotpotQA Cross-Encoder Evaluation

这个 notebook 复用现有的 HotpotQA span 扫描缓存，只针对输入的 span 读取相关文本并做 cross-encoder 预测。

- 不生成 embedding。
- 候选定义来自 `search_wikidata(span, limit=5, include_detailed_description=True, detailed_description_sentences=3, drop_missing_detailed_description=True)`。
- 如果 `search_wikidata` 没有返回候选定义，直接报错并停止。
- 输出展示原始文本上下文以及预测分数最高的定义。


In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")


Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import requests
from IPython.display import display
from sentence_transformers import CrossEncoder

from text_processing import normalize_text
from wikidata_utils import fetch_detailed_descriptions_for_entities


In [3]:
HOTPOT_SCAN_STORE_PATH = Path("hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl")
DEFAULT_MODEL_NAME = "cross-encoder/nli-deberta-v3-large"
DEFAULT_BATCH_SIZE = 32


def load_hotpot_scan_store(scan_store_path: Path = HOTPOT_SCAN_STORE_PATH):
    if not scan_store_path.exists():
        raise FileNotFoundError(
            f"HotpotQA scan store not found at {scan_store_path}. Please prepare the scan cache first."
        )

    with scan_store_path.open("rb") as handle:
        store = pickle.load(handle)

    print(f"Loaded scan-only store from {scan_store_path}")
    print(store["stats"])
    return store


embedding_store = load_hotpot_scan_store()


Loaded scan-only store from hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl
{'num_documents': 66581, 'num_unique_terms': 634624, 'num_phrase_occurrences': 1007780, 'num_token_occurrences': 584999, 'num_total_occurrences': 1592779}


In [4]:
def lookup_records(store, query_text, kind=None, include_text=True, include_cleaned_text=True):
    normalized_query = normalize_text(query_text.strip())
    records = list(store["index"].get(normalized_query, []))

    if kind is not None:
        records = [record for record in records if record["kind"] == kind]

    if not include_text and not include_cleaned_text:
        return records

    enriched_records = []
    for record in records:
        item = dict(record)
        document_idx = item["document_idx"]
        if include_text:
            item["text"] = store["documents"][document_idx]["text"]
        if include_cleaned_text:
            item["cleaned_text"] = store["cleaned_documents"][document_idx]
        enriched_records.append(item)
    return enriched_records


def _find_left_boundary(text: str, index: int) -> int:
    return max(
        text.rfind(".", 0, index),
        text.rfind("!", 0, index),
        text.rfind("?", 0, index),
    )


def _find_right_boundary(text: str, index: int) -> int:
    right_candidates = [
        text.find(".", index),
        text.find("!", index),
        text.find("?", index),
    ]
    right_candidates = [idx for idx in right_candidates if idx != -1]
    return len(text) if not right_candidates else min(right_candidates) + 1


def _build_context_from_bounds(cleaned_text, span, context_start, context_end):
    start_char, end_char = span
    context_raw = cleaned_text[context_start:context_end]

    if not context_raw.strip():
        context_start = max(0, start_char - 120)
        context_end = min(len(cleaned_text), end_char + 120)
        context_raw = cleaned_text[context_start:context_end]

    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - context_start - left_trim
    local_end = end_char - context_start - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {
        "context_text": context_text,
        "local_span": (local_start, local_end),
    }


def extract_sentence_context(cleaned_text, span):
    start_char, end_char = span
    left_boundary = _find_left_boundary(cleaned_text, start_char)
    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = _find_right_boundary(cleaned_text, end_char)
    return _build_context_from_bounds(cleaned_text, span, context_start, context_end)


def extract_neighbor_sentence_context(cleaned_text, span):
    start_char, end_char = span
    left_boundary = _find_left_boundary(cleaned_text, start_char)
    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = _find_right_boundary(cleaned_text, end_char)

    if context_start > 0:
        previous_boundary = _find_left_boundary(cleaned_text, max(0, context_start - 1))
        context_start = 0 if previous_boundary == -1 else previous_boundary + 1

    if context_end < len(cleaned_text):
        context_end = _find_right_boundary(cleaned_text, context_end)

    return _build_context_from_bounds(cleaned_text, span, context_start, context_end)


def extract_full_context(cleaned_text, span):
    start_char, end_char = span
    context_raw = cleaned_text
    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - left_trim
    local_end = end_char - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {
        "context_text": context_text,
        "local_span": (local_start, local_end),
    }


def extract_prompt_context(cleaned_text, span, prompt_context_mode="sentence"):
    if prompt_context_mode == "sentence":
        return extract_sentence_context(cleaned_text, span)
    if prompt_context_mode == "full_text":
        return extract_full_context(cleaned_text, span)
    if prompt_context_mode == "sentence_neighbors":
        return extract_neighbor_sentence_context(cleaned_text, span)
    raise ValueError(
        f"Unsupported prompt_context_mode={prompt_context_mode!r}. Use 'sentence', 'sentence_neighbors', or 'full_text'."
    )


def build_hotpot_prompt(
    record,
    query_text,
    mark_target=False,
    left_marker="[TGT]",
    right_marker="[/TGT]",
    prompt_context_mode="sentence",
):
    context_info = extract_prompt_context(
        record["cleaned_text"],
        record["span"],
        prompt_context_mode=prompt_context_mode,
    )
    context_text = context_info["context_text"]
    local_start, local_end = context_info["local_span"]
    prompt_context = context_text

    if mark_target:
        prompt_context = (
            f"{context_text[:local_start]}{left_marker} {context_text[local_start:local_end]} {right_marker}{context_text[local_end:]}"
        )

    prompt_text = (
        f"Context: {prompt_context}\n"
        f"Target word: {query_text.strip()}\n\n"
        f'Question: What does "{query_text.strip()}" mean in this context?'
    )

    return {
        "context_text": context_text,
        "matched_text": context_text[local_start:local_end],
        "local_span": (local_start, local_end),
        "prompt_text": prompt_text,
    }


In [5]:
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"
DEFAULT_HEADERS = {
    "User-Agent": "WikidataExplorerNotebook/1.0 (https://www.wikidata.org/)"
}


def _safe_get_json(url, params=None, headers=None, timeout=30):
    try:
        response = requests.get(url, params=params, headers=headers or DEFAULT_HEADERS, timeout=timeout)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as exc:
        print(f"Request failed: {exc}")
        return {}
    except ValueError as exc:
        print(f"Invalid JSON response: {exc}")
        return {}


def _coerce_aliases_for_search(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return ", ".join(str(v) for v in value if v is not None)
    if isinstance(value, str):
        return value
    return str(value)


def _series_casefold_equals(series, target):
    target_casefold = str(target).casefold()
    return series.map(lambda value: str(value).casefold() == target_casefold if value is not None else False)

def _series_non_empty_mask(series):
    return series.map(lambda value: bool(str(value).strip()) if value is not None else False)

def _series_exclude_name_descriptions(series):
    blocked_phrases = ("family name", "given name")
    return series.map(
        lambda value: not any(phrase in str(value).casefold() for phrase in blocked_phrases)
        if value is not None else True
    )

def search_wikidata(
    term,
    language="en",
    limit=10,
    exact_match_text=False,
    include_detailed_description=False,
    drop_missing_detailed_description=False,
    detailed_description_sentences=3,
    filter_name=True,
):
    if not isinstance(term, str) or not term.strip():
        print("Please provide a non-empty search term.")
        columns = ["id", "label", "description", "match_text", "aliases", "concepturi"]
        if include_detailed_description:
            columns.insert(3, "detailed_description")
        return pd.DataFrame(columns=columns)

    normalized_term = term.strip()

    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": language,
        "uselang": language,
        "search": normalized_term,
        "limit": int(limit),
    }

    data = _safe_get_json(WIKIDATA_API_URL, params=params)
    items = data.get("search", []) if isinstance(data, dict) else []

    rows = []
    for item in items:
        match = item.get("match") if isinstance(item, dict) else None
        match_text = ""
        if isinstance(match, dict):
            match_text = match.get("text", "")

        row = {
            "id": item.get("id", ""),
            "label": item.get("label", ""),
            "description": item.get("description", ""),
            "match_text": match_text,
            "aliases": _coerce_aliases_for_search(item.get("aliases")),
            "concepturi": item.get("concepturi", ""),
        }
        rows.append(row)

    df = pd.DataFrame(rows, columns=["id", "label", "description", "match_text", "aliases", "concepturi"])

    if filter_name and not df.empty:
        df = df[_series_exclude_name_descriptions(df["description"])].reset_index(drop=True)

    if exact_match_text and not df.empty:
        df = df[_series_casefold_equals(df["match_text"], normalized_term)].reset_index(drop=True)

    if include_detailed_description:
        detailed_descriptions = {}
        if not df.empty:
            detailed_descriptions = fetch_detailed_descriptions_for_entities(
                df["id"].tolist(),
                language=language,
                headers=DEFAULT_HEADERS,
                timeout=30,
                sentences=detailed_description_sentences,
            )
        df.insert(
            df.columns.get_loc("description") + 1,
            "detailed_description",
            [detailed_descriptions.get(entity_id, "") for entity_id in df["id"]],
        )
        if drop_missing_detailed_description:
            df = df[_series_non_empty_mask(df["detailed_description"])].reset_index(drop=True)

    if df.empty:
        print(f"No search results for: {term!r}")
    return df


def load_wikidata_definition_candidates(
    query_text: str,
    use_detailed_description: bool = True,
    exact_match_text: bool = False,
    filter_name: bool = True,
    require_detailed_description: bool = False,
) -> tuple[pd.DataFrame, str]:
    include_detailed_description = use_detailed_description or require_detailed_description
    candidates_df = search_wikidata(
        query_text,
        limit=5,
        exact_match_text=exact_match_text,
        include_detailed_description=include_detailed_description,
        detailed_description_sentences=3,
        drop_missing_detailed_description=include_detailed_description,
        filter_name=filter_name,
    )

    if candidates_df.empty:
        if include_detailed_description:
            raise ValueError(
                f"search_wikidata returned no detailed_description candidates for span={query_text!r}."
            )
        raise ValueError(
            f"search_wikidata returned no description candidates for span={query_text!r}."
        )

    definition_column = "detailed_description" if use_detailed_description else "description"
    candidates_df = candidates_df[_series_non_empty_mask(candidates_df[definition_column])].copy()
    candidates_df = candidates_df.drop_duplicates(subset=["id", definition_column]).reset_index(drop=True)
    if candidates_df.empty:
        raise ValueError(
            f"No usable {definition_column} candidates remained for span={query_text!r}."
        )

    return candidates_df, definition_column


In [6]:
def definition_to_hypothesis(definition: str) -> str:
    cleaned_definition = definition.strip()
    if cleaned_definition.endswith((".", "!", "?")):
        cleaned_definition = cleaned_definition[:-1]
    return f"It refers to {cleaned_definition}."


def extract_cross_encoder_scores(raw_scores, model) -> np.ndarray:
    score_array = np.asarray(raw_scores)
    if score_array.ndim == 1:
        return score_array.astype(float)

    id2label = getattr(model.model.config, "id2label", {}) or {}
    entailment_index = None
    for label_index, label_name in id2label.items():
        if str(label_name).lower() == "entailment":
            entailment_index = int(label_index)
            break

    if entailment_index is None:
        entailment_index = score_array.shape[1] - 1

    return score_array[:, entailment_index].astype(float)


def build_wikidata_candidate_bank(candidates_df: pd.DataFrame, definition_column: str) -> list[dict]:
    candidate_bank = []
    for row in candidates_df.itertuples(index=False):
        definition = str(getattr(row, definition_column)).strip()
        candidate_bank.append(
            {
                "entity_id": row.id,
                "label": row.label,
                "description": row.description,
                "definition_source": definition_column,
                "definition": definition,
                "hypothesis": definition_to_hypothesis(definition),
            }
        )
    return candidate_bank


def evaluate_cross_encoder_on_hotpotqa(
    span_text: str,
    model_name: str = DEFAULT_MODEL_NAME,
    store: dict | None = None,
    kind: str | None = None,
    batch_size: int = DEFAULT_BATCH_SIZE,
    mark_target: bool = False,
    max_records: int | None = None,
    use_detailed_description: bool = True,
    exact_match_text: bool = False,
    prompt_context_mode: str = "sentence",
    require_detailed_description: bool = False,
):
    normalized_span = normalize_text(span_text.strip())
    if not normalized_span:
        raise ValueError("span_text must be a non-empty string.")

    if store is None:
        store = load_hotpot_scan_store()

    records = lookup_records(
        store,
        span_text,
        kind=kind,
        include_text=True,
        include_cleaned_text=True,
    )
    if not records:
        raise ValueError(f"No HotpotQA records were found for span={span_text!r}.")

    if max_records is not None:
        records = records[:max_records]

    candidates_df, definition_column = load_wikidata_definition_candidates(
        span_text.strip(),
        use_detailed_description=use_detailed_description,
        exact_match_text=exact_match_text,
        require_detailed_description=require_detailed_description,
    )
    candidate_bank = build_wikidata_candidate_bank(candidates_df, definition_column=definition_column)
    if not candidate_bank:
        raise ValueError(f"No candidate definitions were built for span={span_text!r}.")

    model = CrossEncoder(model_name)
    evaluation_rows = []

    for record in records:
        prompt_info = build_hotpot_prompt(
            record,
            span_text,
            mark_target=mark_target,
            prompt_context_mode=prompt_context_mode,
        )
        pairs = [(prompt_info["prompt_text"], candidate["hypothesis"]) for candidate in candidate_bank]
        raw_scores = model.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        scores = extract_cross_encoder_scores(raw_scores, model)

        ranked_candidates = sorted(
            [
                {
                    **candidate,
                    "score": float(score),
                }
                for candidate, score in zip(candidate_bank, scores)
            ],
            key=lambda item: item["score"],
            reverse=True,
        )
        top_candidate = ranked_candidates[0]

        evaluation_rows.append(
            {
                "query_span": span_text,
                "normalized_query": normalized_span,
                "title": record["title"],
                "kind": record["kind"],
                "document_idx": record["document_idx"],
                "span": tuple(record["span"]),
                "matched_text": prompt_info["matched_text"],
                "source_text": prompt_info["context_text"],
                "document_text": record["text"],
                "predicted_entity_id": top_candidate["entity_id"],
                "predicted_label": top_candidate["label"],
                "predicted_description": top_candidate["description"],
                "definition_source": top_candidate["definition_source"],
                "predicted_definition": top_candidate["definition"],
                "prediction_score": top_candidate["score"],
                "prompt_text": prompt_info["prompt_text"],
            }
        )

    results_df = pd.DataFrame(evaluation_rows).sort_values(
        ["prediction_score", "title", "document_idx"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    return candidates_df, results_df


def display_hotpot_predictions(results_df: pd.DataFrame, limit: int = 20):
    total_samples = int(len(results_df))

    print(f"Total samples: {total_samples}")

    columns = [
        "query_span",
        "document_text",
        "predicted_description",
        "prediction_score",
    ]
    display(results_df[columns].head(limit))


In [11]:
pd.set_option("display.max_colwidth", None)

TARGET_SPAN = "team"
QUERY_KIND = None
MARK_TARGET = False
PROMPT_CONTEXT_MODE = "sentence_neighbors"   # "sentence"/"sentence_neighbors"/"full_text"
USE_DETAILED_DESCRIPTION = False
REQUIRE_DETAILED_DESCRIPTION = True
EXACT_MATCH_TEXT = True
MAX_RECORDS = 100

candidate_definitions_df, hotpot_results_df = evaluate_cross_encoder_on_hotpotqa(
    span_text=TARGET_SPAN,
    model_name=DEFAULT_MODEL_NAME,
    store=embedding_store,
    kind=QUERY_KIND,
    batch_size=DEFAULT_BATCH_SIZE,
    mark_target=MARK_TARGET,
    max_records=MAX_RECORDS,
    use_detailed_description=USE_DETAILED_DESCRIPTION,
    require_detailed_description=REQUIRE_DETAILED_DESCRIPTION,
    exact_match_text=EXACT_MATCH_TEXT,
    prompt_context_mode=PROMPT_CONTEXT_MODE,
)


In [12]:
definition_column = "detailed_description" if USE_DETAILED_DESCRIPTION else "description"
#display(candidate_definitions_df[["id", "label", "description", definition_column]])
result_columns = [
    "query_span",
    "source_text",
    "predicted_description",
    "prediction_score",
]
hotpot_results_df[result_columns].head(100)


,query_span,source_text,predicted_description,prediction_score
0,team,"after leaving clairefontaine, da silva joined valenciennes and was promoted to its reserve team after two seasons in the clubs youth academy. he helped the team earn promotion to the championnat de france amateur in the 2010–11 season and is the teams current captain. Da Silva made his professional debut with valenciennes on 31 august 2011 against dijon in the coupe de la ligue.",group linked in a common purpose,-1.803702
1,team,the VCU rams finished 6th in the ESPN/USA Today Coaches Poll at the end of the season. this was the highest ranking in VCUs history and the highest ranking of any team from the CAA. the 2011 NCAA tournament run by VCU is regarded as one of the best cinderella runs of all time.,group linked in a common purpose,-1.822645
2,team,"this was the teams first year as the Charlotte Hornets since 2002. the team had been known as the Charlotte Bobcats since its revival in 2004. however, when the team formally changed its name to the hornets on may 20, 2014; they also reclaimed the history and records of the original Charlotte Hornets franchise from the 1988–89 NBA season through the 2001–02 NBA season.",group linked in a common purpose,-1.831446
3,team,"the 20th National Hockey League National Hockey League All-Star Game was played in Montreal Forum on january 18, 1967, where the host Montreal Canadiens defeated a team of all-stars from the remaining NHL teams 3–0. it was the first, and to date, only time a shutout occurred in an all-Star Game.",group linked in a common purpose,-1.887706
4,team,"The Pittsburgh Steelers are a professional American Football team playing in the National Football League. their 1987 season saw the team record an 8-7 record and fail to reach the playoffs. noll was renowned as a stoic character, but in complete contrast was his reaction to Jerry Glanville, the head coach of the oilers.",group linked in a common purpose,-1.894059
...,...,...,...,...
95,team,"in 1921, borough turned professional when their application was accepted to play in the inaugural season of the newly formed Football League Third Division North. the team played in the Football League for ten seasons, with their most successful season coming in 1928–29, finishing fourth in the league and reaching the third round of the FA cup. Wigan Borough folded during the 1931–32 season due to financial problems, and league football did not return to the town until Wigan Athletic f.",group linked in a common purpose,-3.258899
96,team,"the team had been known as the Charlotte Bobcats since its revival in 2004. however, when the team formally changed its name to the hornets on may 20, 2014; they also reclaimed the history and records of the original Charlotte Hornets franchise from the 1988–89 NBA season through the 2001–02 NBA season. The New Orleans Pelicans retained the remaining history that exists under the New Orleans(/Oklahoma City) hornets name from the 2002–03 NBA season through the 2012–13 NBA season.",group linked in a common purpose,-3.281863
97,team,"Parkway Field is the name of a minor league baseball and college baseball park that stood in louisville, kentucky. it was home to the Louisville Colonels of the American Association from 1923 into the mid-1950s, the Louisville Buckeyes of the Negro American League in 1949, and then of the university of louisville team for several decades until they abandoned it in 1998 and moved to Cardinal Stadium. prior to its demolition, Parkway Field had become a home run haven for u of l Head Coach Gene Bakers over the Wall Gang.",group linked in a common purpose,-3.283122
98,team,"the duties of the team manager include team strategy and leadership on and off the field. the team initially began in the now defunct Western League in 1894, and later became one of the American Leagues eight charter franchises in 1901. since the inception of the team in 1894, it has employed 46 different managers.",group linked in a com